In [1]:
import os, re, json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

COMP_DIR = "/kaggle/input/cafa-6-protein-function-prediction"
TRAIN_DIR = os.path.join(COMP_DIR, "Train")
TEST_DIR  = os.path.join(COMP_DIR, "Test")

TRAIN_TERMS = os.path.join(TRAIN_DIR, "train_terms.tsv")
TRAIN_FASTA = os.path.join(TRAIN_DIR, "train_sequences.fasta")
TEST_FASTA  = os.path.join(TEST_DIR,  "testsuperset.fasta")
OBO_PATH    = os.path.join(TRAIN_DIR, "go-basic.obo")

print("files ok:",
      os.path.exists(TRAIN_TERMS),
      os.path.exists(TRAIN_FASTA),
      os.path.exists(TEST_FASTA),
      os.path.exists(OBO_PATH))

files ok: True True True True


**RUN FIRST** - UTILITIES

In [2]:
def normalize_protein_id(x: str) -> str:
    x = str(x).strip().split()[0]
    if "|" in x:
        parts = x.split("|")
        x = parts[1] if len(parts) >= 2 else parts[-1]
    if "." in x:
        left, right = x.rsplit(".", 1)
        if right.isdigit():
            x = left
    return x

def read_fasta_to_df(path: str) -> pd.DataFrame:
    ids, seqs = [], []
    cur_id, cur_seq = None, []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur_id is not None:
                    ids.append(cur_id)
                    seqs.append("".join(cur_seq))
                cur_id = line[1:].split()[0]
                cur_seq = []
            else:
                cur_seq.append(line)
    if cur_id is not None:
        ids.append(cur_id)
        seqs.append("".join(cur_seq))
    return pd.DataFrame({"protein_id": ids, "sequence": seqs})

def make_split_embeddings(pids, X_all, id_to_row):
    rows = []
    kept = []
    for pid in pids:
        r = id_to_row.get(pid, None)
        if r is None:
            continue
        rows.append(r)
        kept.append(pid)
    X = np.asarray(X_all[rows], dtype=np.float32)
    return kept, X

def l2norm(X):
    X = X.astype(np.float32)
    n = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
    return X / n

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

Load sequences + train_terms + aspect normalization

In [3]:
train_seq = read_fasta_to_df(TRAIN_FASTA)
test_seq  = read_fasta_to_df(TEST_FASTA)

train_seq["protein_id"] = train_seq["protein_id"].map(normalize_protein_id)
test_seq["protein_id"]  = test_seq["protein_id"].map(normalize_protein_id)

train_terms = pd.read_csv(TRAIN_TERMS, sep="\t")
if "EntryID" in train_terms.columns:
    train_terms = train_terms.rename(columns={"EntryID":"protein_id", "term":"go_id"})
elif "protein_id" not in train_terms.columns:
    train_terms.columns = ["protein_id","go_id","aspect"]

train_terms["protein_id"] = train_terms["protein_id"].map(normalize_protein_id)
train_terms["aspect"] = train_terms["aspect"].astype(str).str.strip().str.upper()

aspect_map = {
    "P":"BP", "BP":"BP", "BPO":"BP", "BIOLOGICAL_PROCESS":"BP",
    "F":"MF", "MF":"MF", "MFO":"MF", "MOLECULAR_FUNCTION":"MF",
    "C":"CC", "CC":"CC", "CCO":"CC", "CELLULAR_COMPONENT":"CC"
}
train_terms["asp_norm"] = train_terms["aspect"].map(aspect_map)

print("train_seq:", train_seq.shape, "test_seq:", test_seq.shape, "train_terms:", train_terms.shape)
print("asp_norm counts:\n", train_terms["asp_norm"].value_counts(dropna=False))

train_seq: (82404, 2) test_seq: (224309, 2) train_terms: (537027, 4)
asp_norm counts:
 asp_norm
BP    250805
CC    157770
MF    128452
Name: count, dtype: int64


Split train/val

In [4]:
SEED = 42
VAL_FRAC = 0.10
seed_everything(SEED)

seq_to_rep = train_seq.groupby("sequence")["protein_id"].min().rename("rep_id").reset_index()
train_rep = train_seq.merge(seq_to_rep, on="sequence", how="left")

rep_ids = train_rep["rep_id"].drop_duplicates().values
rng = np.random.default_rng(SEED)
rng.shuffle(rep_ids)

n_val = int(len(rep_ids) * VAL_FRAC)
val_rep = set(rep_ids[:n_val])
train_rep_set = set(rep_ids[n_val:])

val_ids   = train_rep.loc[train_rep["rep_id"].isin(val_rep), "protein_id"].unique().tolist()
train_ids = train_rep.loc[train_rep["rep_id"].isin(train_rep_set), "protein_id"].unique().tolist()

print("train:", len(train_ids), "val:", len(val_ids), "overlap:", len(set(train_ids)&set(val_ids)))

train: 74150 val: 8254 overlap: 0


Load ESM2 embeddings

In [5]:
import os
import numpy as np
import pandas as pd

print("Input folders:", os.listdir("/kaggle/input"))

cand = [d for d in os.listdir("/kaggle/input") if "embeddings" in d.lower() and "cafa" in d.lower()]
print("Embedding folder candidates:", cand)
assert len(cand) >= 1, "No embeddings dataset mounted. Click 'Add Input' and add your embeddings dataset."

EMB_DIR = os.path.join("/kaggle/input", cand[0])
print("Using EMB_DIR:", EMB_DIR)
print("Files:", os.listdir(EMB_DIR))

if os.path.exists(os.path.join(EMB_DIR, "protein_embeddings.npy")):
    emb_path = os.path.join(EMB_DIR, "protein_embeddings.npy")
elif os.path.exists(os.path.join(EMB_DIR, "protein_embeddings.npy")):
    emb_path = os.path.join(EMB_DIR, "protein_embeddings.npy")
else:
    npys = [f for f in os.listdir(EMB_DIR) if f.endswith(".npy")]
    assert npys, "No .npy embeddings file found in EMB_DIR."
    emb_path = os.path.join(EMB_DIR, npys[0])

if os.path.exists(os.path.join(EMB_DIR, "protein_ids.csv")):
    id_path = os.path.join(EMB_DIR, "protein_ids.csv")
else:
    csvs = [f for f in os.listdir(EMB_DIR) if f.endswith(".csv")]
    assert csvs, "No .csv id file found in EMB_DIR."
    id_path = os.path.join(EMB_DIR, csvs[0])

print("emb_path:", emb_path)
print("id_path :", id_path)

X_all = np.load(emb_path, mmap_mode="r")
ids_all = pd.read_csv(id_path, header=None)[0].astype(str).map(normalize_protein_id).tolist()

print("X_all:", X_all.shape, "ids:", len(ids_all))
id_to_row = {pid:i for i,pid in enumerate(ids_all)}

Input folders: ['cafa-6-protein-function-prediction', 'cafa6-protein-embeddings-esm2']
Embedding folder candidates: ['cafa6-protein-embeddings-esm2']
Using EMB_DIR: /kaggle/input/cafa6-protein-embeddings-esm2
Files: ['protein_embeddings.npy', 'protein_ids.csv', 'README.md']
emb_path: /kaggle/input/cafa6-protein-embeddings-esm2/protein_embeddings.npy
id_path : /kaggle/input/cafa6-protein-embeddings-esm2/protein_ids.csv
X_all: (287001, 1280) ids: 287002


In [6]:
print("before align  X_all:", X_all.shape, "ids_all:", len(ids_all))

n = min(X_all.shape[0], len(ids_all))
X_all = X_all[:n]
ids_all = ids_all[:n]

print("after  align  X_all:", X_all.shape, "ids_all:", len(ids_all))

id_to_row = {pid: i for i, pid in enumerate(ids_all)}

mx = max(id_to_row.values())
print("max mapped row:", mx, "max allowed:", X_all.shape[0]-1)
assert mx < X_all.shape[0]

before align  X_all: (287001, 1280) ids_all: 287002
after  align  X_all: (287001, 1280) ids_all: 287001
max mapped row: 287000 max allowed: 287000


Align embeds to split + normalize

In [7]:
def make_split_embeddings(pids, X_all, id_to_row):
    rows = []
    kept = []
    for pid in pids:
        r = id_to_row.get(pid)
        if r is None:
            continue
        if r >= X_all.shape[0]:
            continue
        rows.append(r)
        kept.append(pid)
    X = np.asarray(X_all[rows], dtype=np.float32)
    return kept, X

In [8]:
def l2norm(X, eps=1e-12):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(n, eps)

train_pids_aligned, X_train = make_split_embeddings(train_ids, X_all, id_to_row)
val_pids_aligned,   X_val   = make_split_embeddings(val_ids,   X_all, id_to_row)
test_pids_aligned,  X_test  = make_split_embeddings(test_seq["protein_id"].tolist(), X_all, id_to_row)

missing_test = list(set(test_seq["protein_id"]) - set(test_pids_aligned))
print("aligned train/val/test:", len(train_pids_aligned), len(val_pids_aligned), len(test_pids_aligned))
print("missing_test count:", len(missing_test), "example:", missing_test[:5])

X_train = l2norm(X_train)
X_val   = l2norm(X_val)
X_test  = l2norm(X_test)

print("norm check train mean L2:", float(np.mean(np.linalg.norm(X_train, axis=1))))

aligned train/val/test: 74150 8254 224308
missing_test count: 1 example: ['A4QQE0']
norm check train mean L2: 1.0


label vocab + pid→labels per aspect

In [9]:
ASPECTS = ["BP","MF","CC"]

terms_train_split = train_terms[train_terms["protein_id"].isin(train_ids)].copy()

go_vocab = {}
go2i = {}
for asp in ASPECTS:
    vocab = sorted(terms_train_split.loc[terms_train_split["asp_norm"]==asp, "go_id"].unique())
    go_vocab[asp] = vocab
    go2i[asp] = {go:i for i,go in enumerate(vocab)}
    print(asp, "labels:", len(vocab))

def pid2labels_for_aspect(pids_set, asp):
    df = train_terms[(train_terms["protein_id"].isin(pids_set)) & (train_terms["asp_norm"]==asp)][["protein_id","go_id"]].copy()
    df = df[df["go_id"].isin(go2i[asp])]
    df["y"] = df["go_id"].map(go2i[asp]).astype(int)
    return df.groupby("protein_id")["y"].apply(list).to_dict()

train_pid2labels = {asp: pid2labels_for_aspect(set(train_ids), asp) for asp in ASPECTS}
val_pid2labels   = {asp: pid2labels_for_aspect(set(val_ids),   asp) for asp in ASPECTS}

top_fallback = {}
for asp in ASPECTS:
    s = train_terms[train_terms["asp_norm"]==asp]["go_id"].value_counts()
    top_fallback[asp] = s.head(200).index.tolist()
    print(asp, "fallback top1:", top_fallback[asp][0], "len:", len(top_fallback[asp]))

BP labels: 16525
MF labels: 6442
CC labels: 2609
BP fallback top1: GO:0045944 len: 200
MF fallback top1: GO:0005515 len: 200
CC fallback top1: GO:0005634 len: 200


dataset and model

In [10]:
class EmbDataset(Dataset):
    def __init__(self, X, pids, pid2labels):
        self.X = X
        self.pids = pids
        self.pid2labels = pid2labels
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        pid = self.pids[i]
        labs = self.pid2labels.get(pid, [])
        return self.X[i], labs

def collate_emb(batch, n_labels):
    xs, labs_list = zip(*batch)
    Xb = torch.tensor(np.stack(xs), dtype=torch.float32)
    Yb = torch.zeros((len(xs), n_labels), dtype=torch.float32)
    for i, labs in enumerate(labs_list):
        if labs:
            Yb[i, labs] = 1.0
    return Xb, Yb

class MLP2(nn.Module):
    def __init__(self, d_in, d_hid, n_out, p=0.30):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hid),
            nn.BatchNorm1d(d_hid),
            nn.GELU(),
            nn.Dropout(p),
            nn.Linear(d_hid, d_hid),
            nn.BatchNorm1d(d_hid),
            nn.GELU(),
            nn.Dropout(p),
            nn.Linear(d_hid, n_out),
        )
    def forward(self, x): return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
D = X_train.shape[1]

device: cuda


Added some helpers: pos_weight + fmax + temperature search

In [11]:
def make_pos_weight(train_pids, pid2labels, nlab, device):
    pos = np.zeros(nlab, dtype=np.float32)
    for pid in train_pids:
        labs = pid2labels.get(pid, [])
        if labs:
            pos[labs] += 1.0
    N = float(len(train_pids))
    pw = (N - pos) / (pos + 1e-6)
    pw = np.clip(pw, 1.0, 100.0)
    return torch.tensor(pw, dtype=torch.float32, device=device)

def build_multihot(pids, pid2labels, n_labels):
    Y = np.zeros((len(pids), n_labels), dtype=np.uint8)
    for i, pid in enumerate(pids):
        labs = pid2labels.get(pid, [])
        if labs:
            Y[i, labs] = 1
    return Y

def fmax_safe(Y_true, Y_score, thresholds=np.linspace(0.01, 0.99, 99)):
    best_f, best_t = -1.0, 0.20
    for t in thresholds:
        Y_pred = (Y_score >= t).astype(np.uint8)
        tp = (Y_pred & Y_true).sum()
        fp = (Y_pred & (1 - Y_true)).sum()
        fn = ((1 - Y_pred) & Y_true).sum()
        p = tp / (tp + fp + 1e-9)
        r = tp / (tp + fn + 1e-9)
        f = 2 * p * r / (p + r + 1e-9)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_f, best_t

def tune_temperature(val_logits, Y_true):
    temps = np.linspace(0.7, 3.0, 24)
    best = (-1.0, 1.0, 0.20)  # f, T, thr
    for T in temps:
        prob = 1.0 / (1.0 + np.exp(-val_logits / T))
        f, thr = fmax_safe(Y_true, prob)
        if f > best[0]:
            best = (f, float(T), float(thr))
    return best

Train 2-seed ensemble (pos_weight BCE + cosine)

In [12]:
EPOCHS = 10
BATCH = 1024
LR = 2e-3
NUM_WORKERS = 2

ENSEMBLE_SEEDS = [42, 777]

scaler = GradScaler("cuda", enabled=(device.type=="cuda"))

models = {asp: [] for asp in ASPECTS}

for seed in ENSEMBLE_SEEDS:
    seed_everything(seed)
    print("\n=== SEED", seed, "===")

    for asp in ASPECTS:
        nlab = len(go_vocab[asp])

        train_ds = EmbDataset(X_train, train_pids_aligned, train_pid2labels[asp])
        val_ds   = EmbDataset(X_val,   val_pids_aligned,   val_pid2labels[asp])

        train_loader = DataLoader(
            train_ds, batch_size=BATCH, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True,
            collate_fn=lambda b, nlab=nlab: collate_emb(b, nlab)
        )
        val_loader = DataLoader(
            val_ds, batch_size=BATCH, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True,
            collate_fn=lambda b, nlab=nlab: collate_emb(b, nlab)
        )

        model = MLP2(D, 2048, nlab, p=0.30).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

        pos_w = make_pos_weight(train_pids_aligned, train_pid2labels[asp], nlab, device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

        for ep in range(1, EPOCHS+1):
            model.train()
            for Xb, Yb in train_loader:
                Xb = Xb.to(device, non_blocking=True)
                Yb = Yb.to(device, non_blocking=True)

                opt.zero_grad(set_to_none=True)
                with autocast("cuda", enabled=(device.type=="cuda")):
                    logits = model(Xb)
                    loss = criterion(logits, Yb)

                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()

            sched.step()
            print(f"{asp} seed {seed} epoch {ep}/{EPOCHS} done")

        models[asp].append(model.eval())


=== SEED 42 ===
BP seed 42 epoch 1/10 done
BP seed 42 epoch 2/10 done
BP seed 42 epoch 3/10 done
BP seed 42 epoch 4/10 done
BP seed 42 epoch 5/10 done
BP seed 42 epoch 6/10 done
BP seed 42 epoch 7/10 done
BP seed 42 epoch 8/10 done
BP seed 42 epoch 9/10 done
BP seed 42 epoch 10/10 done
MF seed 42 epoch 1/10 done
MF seed 42 epoch 2/10 done
MF seed 42 epoch 3/10 done
MF seed 42 epoch 4/10 done
MF seed 42 epoch 5/10 done
MF seed 42 epoch 6/10 done
MF seed 42 epoch 7/10 done
MF seed 42 epoch 8/10 done
MF seed 42 epoch 9/10 done
MF seed 42 epoch 10/10 done
CC seed 42 epoch 1/10 done
CC seed 42 epoch 2/10 done
CC seed 42 epoch 3/10 done
CC seed 42 epoch 4/10 done
CC seed 42 epoch 5/10 done
CC seed 42 epoch 6/10 done
CC seed 42 epoch 7/10 done
CC seed 42 epoch 8/10 done
CC seed 42 epoch 9/10 done
CC seed 42 epoch 10/10 done

=== SEED 777 ===
BP seed 777 epoch 1/10 done
BP seed 777 epoch 2/10 done
BP seed 777 epoch 3/10 done
BP seed 777 epoch 4/10 done
BP seed 777 epoch 5/10 done
BP seed 777 

Calibrating top-K on VAL

In [13]:
import numpy as np
import torch

TOPK_CAL = 500 
THRS = np.linspace(0.01, 0.70, 140)

def build_multihot(pids, pid2labels, n_labels):
    Y = np.zeros((len(pids), n_labels), dtype=np.uint8)
    for i, pid in enumerate(pids):
        labs = pid2labels.get(pid, [])
        if labs:
            Y[i, labs] = 1
    return Y

def get_model_list(m):
    return m if isinstance(m, (list, tuple)) else [m]

def topk_scores_on_val(models_list, Xv_tensor, k):
    outs_sc = []
    outs_ix = []
    bs = 4096

    for i in range(0, Xv_tensor.shape[0], bs):
        xb = Xv_tensor[i:i+bs].to(device, non_blocking=True)

        prob_sum = None
        with torch.no_grad():
            for mdl in models_list:
                mdl = mdl.to(device).eval()
                p = torch.sigmoid(mdl(xb)).float()
                prob_sum = p if prob_sum is None else (prob_sum + p)
        prob = prob_sum / len(models_list)

        sc, ix = torch.topk(prob, k=k, dim=1)
        outs_sc.append(sc.cpu())
        outs_ix.append(ix.cpu())

    sc = torch.cat(outs_sc, dim=0).numpy()
    ix = torch.cat(outs_ix, dim=0).numpy()
    return sc, ix

def fmax_from_topk(Y_true, topk_sc, topk_ix, thrs):
    best_f, best_t = -1.0, float(thrs[0])

    true_pos_per_row = Y_true.sum(axis=1).astype(np.int64)

    for t in thrs:
        pred_mask = (topk_sc >= t).astype(np.uint8)
        tp = 0
        fp = 0
        pred_pos_per_row = pred_mask.sum(axis=1).astype(np.int64)

        for i in range(topk_ix.shape[0]):
            if pred_pos_per_row[i] == 0:
                continue
            cols = topk_ix[i, pred_mask[i].astype(bool)]
            yrow = Y_true[i, cols]
            tp_i = int(yrow.sum())
            tp += tp_i
            fp += int(len(cols) - tp_i)

        fn = int(true_pos_per_row.sum() - tp)

        p = tp / (tp + fp + 1e-9)
        r = tp / (tp + fn + 1e-9)
        f = 2 * p * r / (p + r + 1e-9)

        if f > best_f:
            best_f, best_t = float(f), float(t)

    return best_f, best_t

thr_aspect = {}

Xv = torch.tensor(X_val, dtype=torch.float32)

for asp in ["BP","MF","CC"]:
    nlab = len(go_vocab[asp])
    k = min(TOPK_CAL, nlab)

    Y_true = build_multihot(val_pids_aligned, val_pid2labels[asp], nlab)

    models_list = get_model_list(models[asp])
    topk_sc, topk_ix = topk_scores_on_val(models_list, Xv, k)

    best_f, thr = fmax_from_topk(Y_true, topk_sc, topk_ix, THRS)
    thr_aspect[asp] = thr

    print(asp, "TOPK_CAL:", k, "best_f:", best_f, "thr:", thr)

thr_aspect

BP TOPK_CAL: 500 best_f: 0.015854466479154512 thr: 0.46172661870503595
MF TOPK_CAL: 500 best_f: 0.07506870386782517 thr: 0.47165467625899277
CC TOPK_CAL: 500 best_f: 0.06669361934862716 thr: 0.4666906474820144


{'BP': 0.46172661870503595,
 'MF': 0.47165467625899277,
 'CC': 0.4666906474820144}

GO ancestor graph

In [14]:
ID_RE = re.compile(r"^id:\s*(GO:\d{7})\s*$")
ISA_RE = re.compile(r"^is_a:\s*(GO:\d{7})\s*!")

def load_go_parents(obo_path):
    parents = {}
    cur = None
    with open(obo_path, "r", encoding="utf-8") as f:
        for line in f:
            m = ID_RE.match(line)
            if m:
                cur = m.group(1)
                parents.setdefault(cur, [])
                continue
            m = ISA_RE.match(line)
            if m and cur is not None:
                parents.setdefault(cur, []).append(m.group(1))
    return parents

GO_PARENTS = load_go_parents(OBO_PATH)
print("GO terms in graph:", len(GO_PARENTS))

from functools import lru_cache

@lru_cache(maxsize=200000)
def get_ancestors(go):
    out = set()
    stack = list(GO_PARENTS.get(go, []))
    while stack:
        p = stack.pop()
        if p in out:
            continue
        out.add(p)
        stack.extend(GO_PARENTS.get(p, []))
    return out

GO terms in graph: 48101


In [15]:
ASPECTS = ["BP", "MF", "CC"]

temp_aspect = {asp: 1.0 for asp in ASPECTS}

print("temp_aspect keys:", temp_aspect.keys())
print("models keys:", models.keys())
print("go_vocab keys:", go_vocab.keys())

temp_aspect keys: dict_keys(['BP', 'MF', 'CC'])
models keys: dict_keys(['BP', 'MF', 'CC'])
go_vocab keys: dict_keys(['BP', 'MF', 'CC'])


Predict TEST + GO-consistent post-process + write submission

In [16]:
TOPK_MODEL = 200
MIN_KEEP_PER_ASP = 3 
KEEP_PER_PROTEIN = 200
ADD_ANCESTORS = True
ANCESTOR_CAP = 0.95 
DEFAULT_FALLBACK_SCORE = 0.10

OUT_PATH = "/kaggle/working/submission.tsv"

Xt = torch.tensor(X_test, dtype=torch.float32)
test_ids = test_pids_aligned
missing_ids = list(set(test_seq["protein_id"]) - set(test_ids))

def probs_ensemble(asp, xb):
    chunk = None
    for m in (models[asp] if isinstance(models[asp], (list, tuple)) else [models[asp]]):
        m = m.to(device).eval()
        with torch.no_grad():
            lg = m(xb)
        chunk = lg if chunk is None else (chunk + lg)

    lg = chunk / (len(models[asp]) if isinstance(models[asp], (list, tuple)) else 1)

    T = float(temp_aspect.get(asp, 1.0))
    prob = torch.sigmoid(lg / T)
    return prob

written = 0
bs = 4096

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for i in range(0, Xt.shape[0], bs):
        xb = Xt[i:i+bs].to(device, non_blocking=True)
        B = xb.shape[0]

        per_protein = [dict() for _ in range(B)]

        with torch.no_grad():
            for asp in ASPECTS:
                nlab = len(go_vocab[asp])
                thr = float(thr_aspect[asp])
                i2go = go_vocab[asp]

                prob = probs_ensemble(asp, xb)  # [B, nlab]
                k = min(TOPK_MODEL, nlab)
                sc, idx = torch.topk(prob, k=k, dim=1)

                sc = sc.cpu().numpy()
                idx = idx.cpu().numpy()

                for r in range(B):
                    kept_any = False
                    for j in range(k):
                        s = float(sc[r, j])
                        if s >= thr:
                            go = i2go[int(idx[r, j])]
                            prev = per_protein[r].get(go, 0.0)
                            if s > prev: per_protein[r][go] = s
                            kept_any = True
                    if not kept_any:
                        for j in range(min(MIN_KEEP_PER_ASP, k)):
                            s = float(sc[r, j])
                            go = i2go[int(idx[r, j])]
                            prev = per_protein[r].get(go, 0.0)
                            if s > prev: per_protein[r][go] = s

        for r in range(B):
            pid = test_ids[i + r]
            d = per_protein[r]

            if ADD_ANCESTORS and d:
                items = list(d.items())
                for go, s in items:
                    anc_s = min(float(s), ANCESTOR_CAP)
                    for a in get_ancestors(go):
                        prev = d.get(a, 0.0)
                        if anc_s > prev:
                            d[a] = anc_s

            if len(d) > KEEP_PER_PROTEIN:
                top = sorted(d.items(), key=lambda x: x[1], reverse=True)[:KEEP_PER_PROTEIN]
            else:
                top = sorted(d.items(), key=lambda x: x[1], reverse=True)

            for go, s in top:
                f.write(f"{pid}\t{go}\t{s:.6f}\n")
                written += 1

    for pid in missing_ids:
        for asp in ASPECTS:
            for go in top_fallback[asp][:50]:
                f.write(f"{pid}\t{go}\t{DEFAULT_FALLBACK_SCORE:.6f}\n")
                written += 1

print("Wrote:", OUT_PATH, "rows:", written, "missing fallback:", missing_ids[:5])

Wrote: /kaggle/working/submission.tsv rows: 32081437 missing fallback: ['A4QQE0']


In [17]:
import pandas as pd

IN_PATH  = "/kaggle/working/submission.tsv"
OUT_PATH = "/kaggle/working/submission_small.tsv"

TOPK_PER_PROTEIN = 200
MIN_SCORE = 0.15 

sub = pd.read_csv(IN_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])
sub["score"] = pd.to_numeric(sub["score"], errors="coerce").fillna(0.0)

sub = sub[sub["score"] >= MIN_SCORE].copy()

sub = sub.sort_values(["protein_id","score"], ascending=[True, False])
sub = sub.groupby("protein_id", as_index=False).head(TOPK_PER_PROTEIN)

sub.to_csv(OUT_PATH, sep="\t", header=False, index=False)
print("wrote:", OUT_PATH, "rows:", len(sub), "avg rows/protein:", sub.groupby("protein_id").size().mean())

wrote: /kaggle/working/submission_small.tsv rows: 32081287 avg rows/protein: 143.02337411059793


In [18]:
sub = pd.read_csv(OUT_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])
print("rows:", len(sub))
print("unique proteins:", sub["protein_id"].nunique(), "expected:", len(test_seq))
print("bad GO:", (~sub["go_id"].astype(str).str.match(r"^GO:\d{7}$")).sum())
sub["score"] = pd.to_numeric(sub["score"], errors="coerce")
print("nan scores:", sub["score"].isna().sum())
print("score min/max:", float(sub["score"].min()), float(sub["score"].max()))
print("dups (pid,go):", sub.duplicated(["protein_id","go_id"]).sum())
print("avg rows/protein:", float(sub.groupby("protein_id").size().mean()))

rows: 32081287
unique proteins: 224308 expected: 224309
bad GO: 0
nan scores: 0
score min/max: 0.395219 0.982481
dups (pid,go): 0
avg rows/protein: 143.02337411059793


In [19]:
!zip -j /kaggle/working/submission_small.zip /kaggle/working/submission_small.tsv
!ls -lh /kaggle/working/submission_small.zip

  adding: submission_small.tsv (deflated 82%)
-rw-r--r-- 1 root root 146M Jan 20 09:29 /kaggle/working/submission_small.zip


In [20]:
import pandas as pd

sub = pd.read_csv("/kaggle/working/submission.tsv", sep="\t", header=None,
                  names=["protein_id","go_id","score"])
cnt = sub.groupby("protein_id").size()

print("proteins:", cnt.shape[0])
print("rows/protein mean:", cnt.mean())
print("rows/protein median:", cnt.median())
print("rows/protein 90%:", cnt.quantile(0.90))
print("rows/protein 99%:", cnt.quantile(0.99))

proteins: 224309
rows/protein mean: 143.0234052133441
rows/protein median: 141.0
rows/protein 90%: 190.0
rows/protein 99%: 200.0


In [21]:
import os
import pandas as pd

IN_PATH  = "/kaggle/working/submission.tsv"
OUT_PATH = "/kaggle/working/submission_small.tsv"

TOPK_PER_PROTEIN = 30 
MIN_SCORE = 0.25 

sub = pd.read_csv(IN_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])
sub["score"] = pd.to_numeric(sub["score"], errors="coerce").fillna(0.0)

sub = sub[sub["score"] >= MIN_SCORE].copy()
sub = sub.sort_values(["protein_id","score"], ascending=[True, False])
sub = sub.groupby("protein_id", as_index=False).head(TOPK_PER_PROTEIN)

expected = set(test_seq["protein_id"].astype(str))
have = set(sub["protein_id"].astype(str))
missing = list(expected - have)

if len(missing) > 0:
    print("Missing proteins after filtering (will add fallback):", missing[:10], "count:", len(missing))
    fallback_terms = ["GO:0005515", "GO:0005634", "GO:0045944"] 
    rows = []
    for pid in missing:
        for go in fallback_terms:
            rows.append((pid, go, float(MIN_SCORE)))
    sub = pd.concat([sub, pd.DataFrame(rows, columns=["protein_id","go_id","score"])], ignore_index=True)

sub.to_csv(OUT_PATH, sep="\t", header=False, index=False)

cnt = sub.groupby("protein_id").size()
print("Wrote:", OUT_PATH)
print("proteins:", cnt.shape[0], "expected:", len(expected))
print("rows:", len(sub))
print("rows/protein mean:", float(cnt.mean()), "median:", float(cnt.median()), "p90:", float(cnt.quantile(0.90)))
print("size MB:", os.path.getsize(OUT_PATH)/1024/1024)

Missing proteins after filtering (will add fallback): ['A4QQE0'] count: 1
Wrote: /kaggle/working/submission_small.tsv
proteins: 224309 expected: 224309
rows: 6729243
rows/protein mean: 29.999879630331375 median: 30.0 p90: 30.0
size MB: 173.07030391693115


In [22]:
import pandas as pd

IN_PATH = "/kaggle/working/submission.tsv"

def make_small(out_path, topk=30, min_score=0.25):
    sub = pd.read_csv(IN_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])
    sub["score"] = pd.to_numeric(sub["score"], errors="coerce").fillna(0.0)

    sub = sub[sub["score"] >= min_score].copy()

    sub = sub.sort_values(["protein_id","score"], ascending=[True, False])
    sub = sub.groupby("protein_id", as_index=False).head(topk)

    if "top_fallback" in globals():
        missing = ["A4QQE0"]
        for pid in missing:
            if (sub["protein_id"] == pid).sum() == 0:
                extra = []
                for asp in ["BP","MF","CC"]:
                    for go in top_fallback[asp][:min(50, len(top_fallback[asp]))]:
                        extra.append((pid, go, float(min_score)))
                sub = pd.concat([sub, pd.DataFrame(extra, columns=["protein_id","go_id","score"])], ignore_index=True)

    sub.to_csv(out_path, sep="\t", header=False, index=False)
    print("wrote:", out_path, "rows:", len(sub), "MB~", round(out_path and (len(sub)*30)/1024/1024, 1))

make_small("/kaggle/working/sub_A.tsv", topk=40, min_score=0.22) 
make_small("/kaggle/working/sub_B.tsv", topk=30, min_score=0.25)
make_small("/kaggle/working/sub_C.tsv", topk=20, min_score=0.30) 

wrote: /kaggle/working/sub_A.tsv rows: 8972430 MB~ 256.7
wrote: /kaggle/working/sub_B.tsv rows: 6729390 MB~ 192.5
wrote: /kaggle/working/sub_C.tsv rows: 4486310 MB~ 128.4


Text predictions

In [23]:
import re

IN_PATH  = "/kaggle/working/submission_small.tsv"   # change if needed
OUT_PATH = "/kaggle/working/submission_with_text.tsv"
OBO_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"

TEXT_LINES_PER_PROTEIN = 3
TERMS_PER_LINE         = 4
MIN_TEXT_PROB          = 0.05
MAX_TEXT_CHARS         = 3000

GO_RE = re.compile(r"^GO:\d{7}$")

def ascii_clean(s: str) -> str:
    s = s.replace("\t", " ")
    s = "".join(ch if 32 <= ord(ch) <= 126 else " " for ch in s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def load_go_meta(obo_path: str):
    go_name = {}
    go_ns = {}
    cur_id = cur_name = cur_ns = None
    with open(obo_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line == "[Term]":
                cur_id = cur_name = cur_ns = None
                continue
            if line.startswith("id: GO:"):
                cur_id = line.split("id: ", 1)[1].strip()
            elif line.startswith("name: "):
                cur_name = line.split("name: ", 1)[1].strip()
            elif line.startswith("namespace: "):
                cur_ns = line.split("namespace: ", 1)[1].strip()
            if cur_id and cur_name and cur_ns:
                go_name[cur_id] = cur_name
                go_ns[cur_id] = cur_ns
    return go_name, go_ns

ns_to_aspect = {
    "biological_process": "BP",
    "molecular_function": "MF",
    "cellular_component": "CC",
}

print("Loading GO names from OBO...")
go_name, go_ns = load_go_meta(OBO_PATH)
print("GO terms loaded:", len(go_name))

def make_text_lines(pid, top_terms):
    byasp = {"BP": [], "MF": [], "CC": []}
    for sc, go in top_terms:
        asp = ns_to_aspect.get(go_ns.get(go, ""), None)
        if asp:
            byasp[asp].append((float(sc), go))

    lines = []
    for asp in ["MF", "BP", "CC"]:
        if not byasp[asp]:
            continue
        picks = byasp[asp][:TERMS_PER_LINE]
        conf = max([sc for sc, _ in picks] + [MIN_TEXT_PROB])
        conf = min(conf, 0.99)

        parts = []
        for sc, go in picks:
            nm = go_name.get(go, "")
            parts.append(f"{nm} ({go})" if nm else go)

        if asp == "MF":
            sent = f"{pid} is predicted to have molecular function(s): " + "; ".join(parts) + "."
        elif asp == "BP":
            sent = f"{pid} is predicted to participate in: " + "; ".join(parts) + "."
        else:
            sent = f"{pid} is predicted to be localized to / part of: " + "; ".join(parts) + "."

        sent = ascii_clean(sent)
        if sent:
            lines.append((conf, sent))
        if len(lines) >= TEXT_LINES_PER_PROTEIN:
            break

    out, total = [], 0
    for conf, sent in lines:
        if total >= MAX_TEXT_CHARS:
            break
        rem = MAX_TEXT_CHARS - total
        sent2 = sent[:rem]
        out.append((conf, sent2))
        total += len(sent2) + 1
    return out

print("Writing combined file:", OUT_PATH)

written_go = 0
written_text = 0

def flush_one(pid, top_terms, fout):
    global written_text
    if pid is None:
        return
    for prob, text in make_text_lines(pid, top_terms):
        fout.write(f"{pid}\tText\t{prob:.3f}\t{text}\n")
        written_text += 1

with open(IN_PATH, "r", encoding="utf-8", errors="ignore") as fin, \
     open(OUT_PATH, "w", encoding="utf-8") as fout:

    cur_pid = None
    cur_top = []

    for raw in fin:
        raw = raw.rstrip("\n")
        if not raw:
            continue
        parts = raw.split("\t")
        if len(parts) < 3:
            continue

        pid = parts[0].strip()
        lab = parts[1].strip()
        scs = parts[2].strip()

        if cur_pid is None:
            cur_pid = pid
        elif pid != cur_pid:
            flush_one(cur_pid, cur_top, fout)
            cur_pid = pid
            cur_top = []

        if GO_RE.match(lab):
            fout.write(f"{pid}\t{lab}\t{scs}\n")
            written_go += 1
            if len(cur_top) < 50:
                try:
                    sc = float(scs)
                except:
                    sc = 0.0
                cur_top.append((sc, lab))
        else:
            # skip any existing Text rows to avoid duplicates
            continue

    flush_one(cur_pid, cur_top, fout)

print("DONE")
print("GO rows written  :", written_go)
print("Text rows written:", written_text)
print("Output:", OUT_PATH)

Loading GO names from OBO...
GO terms loaded: 48101
Writing combined file: /kaggle/working/submission_with_text.tsv
DONE
GO rows written  : 6729243
Text rows written: 556663
Output: /kaggle/working/submission_with_text.tsv


In [24]:
import os
print("MB:", os.path.getsize("/kaggle/working/submission_with_text.tsv")/1024/1024)

MB: 292.6517086029053


**trying to improve the score using knn + mlp**

L2 normalize

In [35]:
import numpy as np

def l2norm(X, eps=1e-12):
    X = X.astype(np.float32, copy=False)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (n + eps)

X_train_n = l2norm(X_train)
X_val_n   = l2norm(X_val)
X_test_n  = l2norm(X_test)

print("norm check (train mean):", float(np.mean(np.linalg.norm(X_train_n, axis=1))))
print("norm check (test  mean):", float(np.mean(np.linalg.norm(X_test_n, axis=1))))

norm check (train mean): 1.0
norm check (test  mean): 1.0


MLP ensemble probability

In [42]:
import torch

@torch.no_grad()
def probs_ensemble(models_dict, asp, xb_torch, device, T=1.0):
    """
    xb_torch: torch.FloatTensor [B, D] on device
    returns: torch.FloatTensor [B, n_labels] on CPU
    """
    ms = models_dict[asp]
    if isinstance(ms, (list, tuple)):
        logits = None
        for m in ms:
            m = m.to(device).eval()
            lg = m(xb_torch)
            logits = lg if logits is None else (logits + lg)
        logits = logits / float(len(ms))
    else:
        m = ms.to(device).eval()
        logits = m(xb_torch)

    logits = logits / float(T)
    return torch.sigmoid(logits).float().cpu()

bulding train_go_lists[asp][row] = list of GO IDs for that train protein row 

In [40]:
ASPECTS = ["BP", "MF", "CC"]

pid_to_idx_train = {pid: i for i, pid in enumerate(train_pids_aligned)}

train_go_lists = {}
for asp in ASPECTS:
    i2go = go_vocab[asp]                 # index -> GO string
    pid2labs = train_pid2labels[asp]     # pid -> list of label indices

    lists = [[] for _ in range(len(train_pids_aligned))]
    for pid, irow in pid_to_idx_train.items():
        labs = pid2labs.get(pid, [])
        if labs:
            lists[irow] = [i2go[j] for j in labs]
    train_go_lists[asp] = lists

print("Built train_go_lists.")
for asp in ASPECTS:
    ex = train_go_lists[asp][0][:5]
    print(asp, "example first row:", ex, "| total rows:", len(train_go_lists[asp]))

Built train_go_lists.
BP example first row: ['GO:0001649', 'GO:0033687', 'GO:0043610', 'GO:2001145', 'GO:0032147'] | total rows: 74150
MF example first row: ['GO:0003677', 'GO:0140297'] | total rows: 74150
CC example first row: ['GO:0005615', 'GO:0005634', 'GO:0005739'] | total rows: 74150


kNN helpers

In [44]:
@torch.no_grad()
def knn_predict_batch(Xb_np, asp, KNN_K=30, use_power=2.0):
    """
    Xb_np: (B, D) numpy float32, L2-normalized
    returns: list[dict] length B mapping go->score
    """
    Xtr = torch.from_numpy(X_train_n).to(device)          # (Ntr, D)
    Xb  = torch.from_numpy(Xb_np).to(device)              # (B, D)

    sim = Xb @ Xtr.T                                      # (B, Ntr)
    topv, topi = torch.topk(sim, k=KNN_K, dim=1)          # (B, K)

    topv = topv.float().cpu().numpy()
    topi = topi.cpu().numpy()

    go_lists = train_go_lists[asp]

    out = []
    for r in range(topi.shape[0]):
        d = {}
        for j in range(topi.shape[1]):
            s = float(topv[r, j])
            if s <= 0:
                continue
            w = (s ** use_power)
            for go in go_lists[int(topi[r, j])]:
                d[go] = d.get(go, 0.0) + w
        out.append(d)
    return out

MLP topK + normalized kNN dict

In [46]:
def mix_mlp_knn_dict(asp, mlp_sc, mlp_idx, knn_dicts, alpha=0.6):
    """
    mlp_sc/mlp_idx: numpy arrays from topk over MLP probs
    knn_dicts: list of dict go->score
    returns: list[dict] go->mixed_score
    """
    i2go = go_vocab[asp]
    out = []

    for r in range(mlp_sc.shape[0]):
        d = {}

        # MLP contribution
        for j in range(mlp_sc.shape[1]):
            go = i2go[int(mlp_idx[r, j])]
            d[go] = d.get(go, 0.0) + alpha * float(mlp_sc[r, j])

        # kNN contribution (normalize per protein)
        kd = knn_dicts[r]
        if kd:
            mx = max(kd.values())
            if mx > 0:
                w = (1.0 - alpha)
                for go, v in kd.items():
                    d[go] = d.get(go, 0.0) + w * float(v / mx)

        out.append(d)

    return out

Define defaults

In [47]:
ASPECTS = ["BP","MF","CC"]

# Mixing weights (good starting point)
alpha_aspect = {"BP": 0.65, "MF": 0.55, "CC": 0.55}

# If you don’t have calibrated thresholds for the *mixed* output yet:
thr_aspect = thr_aspect if "thr_aspect" in globals() else {"BP": 0.20, "MF": 0.25, "CC": 0.25}

# Optional temperature scaling (ok if not defined)
temp_aspect = temp_aspect if "temp_aspect" in globals() else {"BP": 1.0, "MF": 1.0, "CC": 1.0}

submision file

In [48]:
import numpy as np
import torch

OUT_PATH = "/kaggle/working/submission_ensemble.tsv"

MAX_TOTAL_TERMS_PER_PROTEIN = 1500
TOP_PER_ASPECT = {"BP": 400, "MF": 400, "CC": 400}
MIN_SCORE_FLOOR = 1e-6

TEST_BATCH = 512
KNN_K = 30
MLP_TOPK = 200

def fmt_3sig(x: float) -> str:
    x = float(x)
    if x <= 0:
        return None
    if x > 1:
        x = 1.0
    return f"{x:.3g}"

missing_ids = list(set(test_seq["protein_id"]) - set(test_pids_aligned))
print("missing embeddings ids:", missing_ids[:5], "count:", len(missing_ids))

written = 0
with open(OUT_PATH, "w", encoding="utf-8") as f:

    for i0 in range(0, X_test_n.shape[0], TEST_BATCH):
        xb_np = X_test_n[i0:i0+TEST_BATCH]                  # numpy (B,D)
        xb_t  = torch.from_numpy(xb_np).to(device)          # torch (B,D) on GPU

        batch_preds = {asp: None for asp in ASPECTS}

        for asp in ASPECTS:
            thr = float(thr_aspect.get(asp, 0.20))
            alpha = float(alpha_aspect.get(asp, 0.6))
            T = float(temp_aspect.get(asp, 1.0))

            # kNN dicts
            knn = knn_predict_batch(xb_np, asp, KNN_K=KNN_K)

            # MLP probs (ensemble-safe)
            prob = probs_ensemble(models, asp, xb_t, device, T=T)     # torch on CPU, (B, nlab)

            # topK over MLP probs
            k = min(MLP_TOPK, prob.shape[1])
            sc, idx = torch.topk(prob, k=k, dim=1)
            sc = sc.numpy()
            idx = idx.numpy()

            pred = mix_mlp_knn_dict(asp, sc, idx, knn, alpha=alpha)

            # threshold + keep top per aspect
            out = []
            for d in pred:
                items = [(go, s) for go, s in d.items() if (s >= thr and s > MIN_SCORE_FLOOR)]
                items.sort(key=lambda x: x[1], reverse=True)
                out.append(items[:TOP_PER_ASPECT[asp]])
            batch_preds[asp] = out

        # write with combined cap
        B = xb_np.shape[0]
        for r in range(B):
            pid = test_pids_aligned[i0 + r]

            combined = []
            for asp in ASPECTS:
                combined.extend(batch_preds[asp][r])

            combined.sort(key=lambda x: x[1], reverse=True)
            combined = combined[:MAX_TOTAL_TERMS_PER_PROTEIN]

            for go, s in combined:
                s_txt = fmt_3sig(s)
                if s_txt is None:
                    continue
                f.write(f"{pid}\t{go}\t{s_txt}\n")
                written += 1

    # fallback for missing proteins
    if missing_ids:
        for pid in missing_ids:
            for asp in ASPECTS:
                for go in top_fallback[asp][:50]:
                    s_txt = fmt_3sig(0.1)
                    f.write(f"{pid}\t{go}\t{s_txt}\n")
                    written += 1

print("Wrote:", OUT_PATH, "rows:", written)

missing embeddings ids: ['A4QQE0'] count: 1
Wrote: /kaggle/working/submission_ensemble.tsv rows: 2980459


verifications:

In [49]:
import os, pandas as pd

print("MB:", os.path.getsize(OUT_PATH)/1024/1024)

df = pd.read_csv(OUT_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])
print("proteins:", df["protein_id"].nunique(), "expected:", len(test_seq))
print("max terms/protein:", int(df.groupby("protein_id").size().max()))
print("min/max score:", float(df["score"].min()), float(df["score"].max()))

MB: 68.13507556915283
proteins: 224309 expected: 224309
max terms/protein: 171
min/max score: 0.1 0.92


In [50]:
import pandas as pd, re, numpy as np

df = pd.read_csv(OUT_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])

# 1) GO id format
bad_go = (~df["go_id"].astype(str).str.match(r"^GO:\d{7}$")).sum()
print("bad GO:", bad_go)

# 2) duplicates (protein_id, go_id)
dups = df.duplicated(["protein_id","go_id"]).sum()
print("duplicate rows:", dups)

# 3) score range + NaNs
df["score"] = pd.to_numeric(df["score"], errors="coerce")
print("NaN scores:", df["score"].isna().sum())
print("out of (0,1]:", ((df["score"] <= 0) | (df["score"] > 1)).sum())

bad GO: 0
duplicate rows: 0
NaN scores: 0
out of (0,1]: 0


In [51]:
import pandas as pd

p = "/kaggle/working/submission.tsv"
df = pd.read_csv(p, sep="\t", header=None, names=["protein_id","go_id","score"])
df["score"] = pd.to_numeric(df["score"], errors="coerce")
cnt = df.groupby("protein_id").size()

print("rows:", len(df))
print("proteins:", df["protein_id"].nunique())
print("avg terms/protein:", float(cnt.mean()))
print("median:", float(cnt.median()), "p90:", float(cnt.quantile(0.9)))
print("score min/mean/max:", float(df["score"].min()), float(df["score"].mean()), float(df["score"].max()))

rows: 32081437
proteins: 224309
avg terms/protein: 143.0234052133441
median: 141.0 p90: 190.0
score min/mean/max: 0.1 0.5400863344984197 0.982481
